# ENN583 Week 11 — Modern Transformer Segmentation

This companion practical introduces **Mask2Former**, a transformer-based model for image segmentation. We will use a small model trained for instance segmentation on COCO and compare its behaviour with Mask R-CNN from the main practical.

Mask2Former predicts a set of object masks and class labels. Unlike Mask R-CNN, it uses transformer queries and masked attention rather than proposing regions and then classifying them.

## Learning objectives

By the end of this notebook, you should be able to:

1. Distinguish semantic, instance, and panoptic segmentation.
2. Run a pretrained Mask2Former model using a Hugging Face pipeline.
3. Interpret the label, confidence score, and mask returned for each instance.
4. Investigate how confidence and mask thresholds affect the result.
5. Compare Mask2Former qualitatively with Mask R-CNN.

## 1. Setup

The first execution downloads the pretrained model from Hugging Face and therefore requires internet access. Later runs use the local Hugging Face cache. The small checkpoint is used to keep the practical suitable for limited-memory teaching machines.

In [ ]:
from pathlib import Path
import glob

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
print('Using:', 'CUDA' if device == 0 else 'CPU')
if device == 0:
    print(torch.cuda.get_device_name(0))

In [ ]:
def draw_segments(image, segments, title):
    # Draw each predicted mask over the original image.
    image_array = np.asarray(image).astype(float) / 255
    overlay = image_array.copy()
    colours = plt.cm.tab10.colors

    fig, ax = plt.subplots(figsize=(10, 7))

    for index, segment in enumerate(segments):
        mask = np.asarray(segment['mask']) > 0
        colour = np.asarray(colours[index % len(colours)])
        overlay[mask] = 0.5 * overlay[mask] + 0.5 * colour

        rows, columns = np.where(mask)
        if len(rows) > 0:
            label = f"{segment['label']}: {segment['score']:.2f}"
            ax.text(
                columns.min(), rows.min(), label, color='white', fontsize=9,
                bbox={'facecolor': colour, 'alpha': 0.85, 'pad': 2}
            )

    ax.imshow(overlay)
    ax.set_title(title)
    ax.axis('off')
    plt.show()

In [ ]:
# Reuse an image from the Week 10 object-detection practical.
image_path = Path('../Week_10/images/coco/000000017029.jpg')
image = Image.open(image_path).convert('RGB')
image

## 2. Instance segmentation with Mask2Former

The Hugging Face image-segmentation pipeline handles image preprocessing, model inference, and conversion of the raw query predictions into masks. We explicitly request the `instance` subtask because we want a separate mask for every detected object.

Each returned dictionary contains:

- `label`: the predicted COCO class;
- `score`: the confidence assigned to the instance; and
- `mask`: a binary PIL image identifying its pixels.

In [ ]:
model_id = 'facebook/mask2former-swin-small-coco-instance'
segmenter = pipeline(
    'image-segmentation',
    model=model_id,
    device=device,
    dtype=torch.float16 if device == 0 else torch.float32,
)

segments = segmenter(
    image, subtask='instance', threshold=0.5, mask_threshold=0.5
)

print(f'Found {len(segments)} instances')
for segment in segments:
    print(segment['label'], f"{segment['score']:.3f}", segment['mask'].size)

In [ ]:
draw_segments(image, segments, 'Mask2Former — confidence threshold = 0.5')

## 3. Confidence and mask thresholds

Mask2Former exposes two useful thresholds:

- The **confidence threshold** controls which predicted instances are retained. Lowering it usually reveals more objects, but can also introduce false positives.
- The **mask threshold** converts mask probabilities into foreground/background pixels. Lowering it generally grows a mask, while raising it makes the mask more conservative.

### Your turn: vary the thresholds

Run the model with lower and higher confidence or mask thresholds. Compare the number of retained instances and the shapes of their masks.

In [ ]:
# TODO: Implement this solution.
pass


## 4. Test the reused COCO images

### Your turn: look for successes and failures

Run Mask2Former on several images from the Week 10 COCO collection. Look for:

- dogs that are missed or assigned the wrong label;
- separate objects that are merged into one mask;
- a single object that is split into multiple masks; and
- differences between the Mask2Former and Mask R-CNN results.



In [ ]:
# TODO: Implement this solution.
pass


## 5. Comparison and discussion

| Model | Main idea | Output | Practical observation |
|---|---|---|---|
| Mask R-CNN | Region proposals followed by classification, box regression, and mask prediction | Boxes, labels, scores, and masks | Strong conventional instance-segmentation baseline |
| Mask2Former | Transformer queries with masked attention | Labels, scores, and masks | Modern unified approach to segmentation |

Discuss the following:

1. Which model produced cleaner boundaries around the dog?
2. Did both models detect the same object instances?
3. How did the confidence and mask thresholds affect different aspects of the output?
4. Why should confidence scores not be compared as though they have exactly the same meaning across models?
5. Mask2Former can also be trained for semantic and panoptic segmentation. How do those tasks differ from the instance segmentation shown here?

## If a 4 GB GPU runs out of memory

1. Restart the kernel and run only this notebook.
2. Close other GPU applications and notebooks.
3. Resize very large images before passing them to the pipeline.
4. Set `device = -1` to run on CPU as a slower fallback.
5. Process one image at a time rather than passing a batch.